# Experiment 2.5: Does Muon's Gauge-Direction Chaos Make It Worse for Fine-Tuning?

## Theoretical Motivation

The gauge-fixing interpretation of Muon posits that Newton-Schulz orthogonalization
projects gradient updates onto the orthogonal group O(n), effectively "fixing the gauge"
of the weight-space symmetry inherent in deep linear networks. This projection has a
dual character:

- **Exploration benefit**: By forcing updates along orthogonal directions, Muon explores
  the loss landscape more uniformly, avoiding the slow convergence along near-degenerate
  directions that plagues vanilla SGD. This is the mechanism behind Muon's superior
  training-from-scratch performance.

- **Exploitation cost**: The same orthogonal projection injects "chaos" into the
  optimization trajectory. Near a pre-trained minimum, the loss landscape is approximately
  quadratic, and the optimal strategy is to follow the local Hessian curvature (i.e.,
  gentle gradient descent). Muon's Newton-Schulz step discards magnitude information
  and replaces it with an orthogonal rotation, which destabilizes the trajectory
  relative to the basin geometry. In dynamical-systems language, Muon has a higher
  Lyapunov exponent near fixed points.

## Central Hypothesis

**Muon is better for exploration (training from scratch) but worse for exploitation
(fine-tuning from a pre-trained checkpoint).** The Newton-Schulz orthogonalization
injects gauge-direction chaos that destabilizes the basin around a pre-trained minimum,
whereas SGD can gently slide into the nearest good minimum.

## Experimental Protocol

| Phase | Description | Steps | Optimizer |
|-------|-------------|-------|-----------|
| **Phase 1** | Pre-train a 4-layer deep linear net (32x32) on W_target with SGD | 500 | SGD only |
| **Phase 2** | Modify 20% of W_target, then fine-tune from checkpoint | 200 | SGD vs Muon |
| **Phase 3** | Train from random init on modified target | 700 | SGD vs Muon |

## Three Testable Predictions

1. **[H1] Distance from checkpoint**: Muon wanders further from the pre-trained weights
   during fine-tuning (higher ||W_t - W_ckpt||_F), reflecting chaotic trajectory dynamics.
2. **[H2] Fine-tuning loss**: SGD achieves lower final loss than Muon when fine-tuning,
   because it exploits the nearby basin without disrupting it.
3. **[H3] From-scratch loss**: Muon achieves lower final loss than SGD when training
   from scratch, because its exploration advantage dominates when there is no basin
   to exploit.

## Why This Matters for the Gauge-Fixing Theory

If confirmed, this experiment demonstrates that Muon's gauge-fixing mechanism has a
**regime-dependent** character: it is beneficial when the optimization problem requires
breaking symmetry (early training), but harmful when the problem requires preserving
a particular symmetry-broken state (fine-tuning). This asymmetry is a direct prediction
of the RG gauge-fixing framework that would NOT follow from a naive "better optimizer"
narrative.

In [ ]:
"""
Exp 2.5: Muon worse for fine-tuning -- chaos prevents settling near pre-trained minimum
========================================================================================

HYPOTHESIS:
  Muon's higher Lyapunov exponent means it is better for exploration (training
  from scratch) but worse for exploitation (fine-tuning from a checkpoint).
  The Newton-Schulz orthogonalization injects gauge-direction chaos that
  destabilises the basin around a pre-trained minimum, whereas SGD can gently
  slide into the nearest good minimum.

PROTOCOL:
  Phase 1 -- Pre-training (shared):
      Train a 4-layer deep linear net (32x32) from random init with SGD
      for 500 steps on target W_target.  Save checkpoint.

  Phase 2 -- Fine-tuning (from checkpoint):
      Modify 20% of the target matrix -> W_target_modified.
      From the checkpoint, fine-tune 200 steps with:
        (a) SGD   lr=0.01
        (b) Muon  lr=0.005

  Phase 3 -- From-scratch comparison:
      Train from random init for 700 steps on W_target_modified with both
      optimizers.

MEASUREMENTS:
  - Fine-tuning loss curves  (SGD vs Muon)
  - From-scratch loss curves  (SGD vs Muon)
  - Distance from checkpoint  ||W_t - W_checkpoint||_F  for fine-tuning runs

PREDICTIONS:
  - Fine-tuning: SGD stays closer to checkpoint, Muon wanders further
  - Fine-tuning final loss: SGD < Muon (SGD exploits the nearby minimum)
  - From-scratch final loss: Muon < SGD (Muon explores better)
"""

## Setup and Imports

We use only NumPy to keep the experiment transparent and reproducible.
No deep learning frameworks are needed -- deep linear networks admit
exact gradient computation via straightforward matrix calculus.

In [ ]:
import numpy as np
import os

## Experimental Configuration

### Design Rationale for Hyperparameters

The configuration below is chosen to create a controlled test of the
exploration-vs-exploitation trade-off:

- **DIM=32, NUM_LAYERS=4**: A 4-layer deep linear network with 32x32 weight matrices.
  Deep linear networks are the simplest setting where the gauge symmetry (simultaneous
  invertible transformation at each layer boundary) is exactly present. Four layers
  provide enough depth for the gauge orbit to be non-trivial.

- **PRETRAIN_STEPS=500**: Long enough for SGD to converge near the original target,
  establishing a well-defined basin. The checkpoint must represent a genuine minimum,
  not a transient.

- **FINETUNE_STEPS=200**: Short enough to test whether the optimizer can quickly adapt
  to a modified target without losing the benefit of pre-training.

- **MODIFY_FRAC=0.20**: 20% of target entries are changed -- enough to require
  non-trivial adaptation, but not so much that the pre-trained basin becomes irrelevant.

- **MUON_FT_LR=0.005 vs SGD_FT_LR=0.01**: Muon's learning rate is halved because
  the orthogonal update has unit spectral norm (by construction), while SGD's raw
  gradient magnitude varies. This prevents trivial instability from LR mismatch.

- **NUM_RUNS=5**: We average over multiple random seeds to ensure results are not
  artifacts of a single initialization.

In [ ]:
SEED = 42
np.random.seed(SEED)

In [ ]:
DIM = 32
NUM_LAYERS = 4
BATCH_SIZE = 64

In [ ]:
# Phase 1 -- pre-training
PRETRAIN_STEPS = 500
PRETRAIN_LR = 0.01

In [ ]:
# Phase 2 -- fine-tuning
FINETUNE_STEPS = 200
SGD_FT_LR = 0.01
MUON_FT_LR = 0.005

In [ ]:
# Phase 3 -- from scratch
SCRATCH_STEPS = 700
SGD_SCRATCH_LR = 0.01
MUON_SCRATCH_LR = 0.005

In [ ]:
# Muon parameters
MOMENTUM = 0.9
NS_ITERS = 5

In [ ]:
# Target modification fraction
MODIFY_FRAC = 0.20

In [ ]:
# Number of runs to average over for robustness
NUM_RUNS = 5

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

In [ ]:
# Fixed data
X_data = np.random.randn(DIM, BATCH_SIZE) * 0.3

In [ ]:
# --- Diagnostic: Print full configuration for reproducibility ---
print("=" * 80)
print("EXPERIMENT 2.5: CONFIGURATION SUMMARY")
print("=" * 80)
print(f"  Network:           {NUM_LAYERS}-layer deep linear, {DIM}x{DIM} weight matrices")
print(f"  Total parameters:  {NUM_LAYERS * DIM * DIM} (= {NUM_LAYERS} x {DIM}x{DIM})")
print(f"  Batch size:        {BATCH_SIZE}")
print(f"  Random seed:       {SEED}")
print()
print(f"  Phase 1 (pre-train):   {PRETRAIN_STEPS} steps, SGD lr={PRETRAIN_LR}")
print(f"  Phase 2 (fine-tune):   {FINETUNE_STEPS} steps, SGD lr={SGD_FT_LR}, Muon lr={MUON_FT_LR}")
print(f"  Phase 3 (from scratch):{SCRATCH_STEPS} steps, SGD lr={SGD_SCRATCH_LR}, Muon lr={MUON_SCRATCH_LR}")
print(f"  Target modification:   {MODIFY_FRAC*100:.0f}% of entries replaced")
print(f"  Muon NS iterations:    {NS_ITERS}")
print(f"  Momentum (both):       {MOMENTUM}")
print(f"  Number of runs:        {NUM_RUNS}")
print()
print(f"  Input data shape:      X_data is {DIM}x{BATCH_SIZE}, scaled by 0.3")
print(f"  Input data norm:       ||X_data||_F = {np.linalg.norm(X_data):.4f}")
print("=" * 80)

## Target Matrices

### The Role of Target Modification

The original target matrix W_target represents the "pre-training task."
During fine-tuning, we modify 20% of its entries to simulate a
distribution shift -- the kind of scenario where practitioners
typically fine-tune rather than retrain from scratch.

The key question: does the pre-trained checkpoint help, and does the
choice of optimizer (SGD vs Muon) affect how well that help is utilized?

The Frobenius distance between W_target_original and W_target_modified
measures the "size" of the distribution shift. If this distance is small
relative to the basin width, fine-tuning should be effective. If Muon's
chaotic dynamics push the weights further than necessary, the pre-training
benefit is wasted.

In [ ]:
W_target_original = np.random.randn(DIM, DIM) * 0.5

In [ ]:
def make_modified_target(W_original, frac, rng):
    """Change `frac` of entries to new random values."""
    W_mod = W_original.copy()
    n_entries = W_mod.size
    n_change = int(frac * n_entries)
    indices = rng.choice(n_entries, size=n_change, replace=False)
    flat = W_mod.ravel()
    flat[indices] = rng.randn(n_change) * 0.5
    return W_mod

In [ ]:
# --- Diagnostic: Inspect the original target matrix ---
print("TARGET MATRIX PROPERTIES")
print("-" * 50)
print(f"  W_target_original shape:       {W_target_original.shape}")
print(f"  W_target_original Frobenius norm: {np.linalg.norm(W_target_original, 'fro'):.4f}")
print(f"  W_target_original spectral norm:  {np.linalg.norm(W_target_original, 2):.4f}")
print(f"  W_target_original condition number: {np.linalg.cond(W_target_original):.2f}")
_U, _S, _Vt = np.linalg.svd(W_target_original)
print(f"  Singular values (top-5):  {_S[:5].round(4)}")
print(f"  Singular values (bot-5):  {_S[-5:].round(4)}")
print(f"  Rank (tol=1e-10):         {np.linalg.matrix_rank(W_target_original)}")
print()
# Preview modification magnitude
_rng_preview = np.random.RandomState(SEED)
_W_mod_preview = make_modified_target(W_target_original, MODIFY_FRAC, _rng_preview)
_shift_norm = np.linalg.norm(_W_mod_preview - W_target_original, 'fro')
print(f"  Example modified target shift: ||W_mod - W_orig||_F = {_shift_norm:.4f}")
print(f"  Relative shift:  {_shift_norm / np.linalg.norm(W_target_original, 'fro') * 100:.1f}% of ||W_orig||_F")
print(f"  Number of entries modified:     {int(MODIFY_FRAC * W_target_original.size)} / {W_target_original.size}")
del _rng_preview, _W_mod_preview, _shift_norm, _U, _S, _Vt

## Network Architecture and Helpers

### Deep Linear Networks as a Gauge Theory Test Bed

A deep linear network computes f(X) = W_L ... W_2 W_1 X. Despite being
"linear" in the end-to-end sense (the product W_L...W_1 is a single matrix),
the loss landscape is highly non-convex due to the **gauge symmetry**:
for any invertible matrix G, the transformation W_l -> W_l G, W_{l-1} -> G^{-1} W_{l-1}
leaves the network function unchanged.

This means the loss landscape has continuous families of equivalent minima
(gauge orbits), and the optimizer must navigate both the "physical" directions
(that change the network function) and the "gauge" directions (that merely
reparametrize the same function).

**Muon's orthogonal projection acts as a gauge-fixing mechanism**: it constrains
updates to orthogonal matrices, which breaks the GL(n) gauge symmetry down
to O(n). The question for fine-tuning is whether this gauge-fixing is helpful
or harmful when we already have a good solution.

### Initialization Strategy

Weights are initialized near the identity: W_l = I + 0.1 * noise.
This ensures the initial end-to-end map is close to the identity,
avoiding exploding/vanishing gradients in the deep linear chain
while still providing enough randomness for training dynamics to
break symmetry.

In [ ]:
def init_weights(num_layers, rng):
    """Initialize layers near identity for stability."""
    weights = []
    for _ in range(num_layers):
        W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
        weights.append(W.copy())
    return weights

In [ ]:
def forward_linear(weights, X):
    """Forward pass through deep linear net."""
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y_target):
    """Quadratic loss: 0.5 * ||f(X) - Y||^2 / N."""
    Y_pred = forward_linear(weights, X)
    diff = Y_pred - Y_target
    return 0.5 * np.mean(np.sum(diff**2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y_target):
    """Backprop through deep linear net for quadratic loss."""
    num_layers = len(weights)
    N = X.shape[1]

    # Forward pass -- store activations
    activations = [X.copy()]
    for W in weights:
        activations.append(W @ activations[-1])

    # Output error
    delta = (activations[-1] - Y_target) / N

    # Backward pass
    grads = [None] * num_layers
    for l in range(num_layers - 1, -1, -1):
        grads[l] = delta @ activations[l].T
        if l > 0:
            delta = weights[l].T @ delta

    return grads

In [ ]:
def flatten_weights(weights):
    return np.concatenate([W.ravel() for W in weights])

In [ ]:
def checkpoint_distance(weights, checkpoint_weights):
    """Frobenius distance between current weights and checkpoint."""
    flat_w = flatten_weights(weights)
    flat_c = flatten_weights(checkpoint_weights)
    return np.linalg.norm(flat_w - flat_c)

In [ ]:
def copy_weights(weights):
    return [W.copy() for W in weights]

## Optimizers: SGD vs Muon

### SGD with Momentum

Standard SGD with momentum accumulates a velocity buffer:
  v_t = beta * v_{t-1} + grad_t,  W_t = W_{t-1} - lr * v_t

The update direction is determined by the raw gradient and its history.
Near a minimum, the gradient is small and approximately aligned with
the Hessian eigenvectors, so SGD follows the local curvature -- exactly
the behavior we want for fine-tuning.

### Muon (Momentum + Newton-Schulz Orthogonalization)

Muon replaces the velocity buffer with its orthogonal polar factor:
  v_t = beta * v_{t-1} + grad_t,  U_t = NS_ortho(v_t),  W_t = W_{t-1} - lr * U_t

The Newton-Schulz iteration approximates U_t = V_t (V_t^T V_t)^{-1/2},
the closest orthogonal matrix to the velocity. This has two key effects:

1. **Magnitude equalization**: All singular values of U_t are 1, so the update
   magnitude is independent of the gradient magnitude. Near a minimum where
   gradients are small, this means Muon takes disproportionately large steps
   compared to what the gradient landscape suggests.

2. **Direction rotation**: The orthogonal projection rotates the update direction
   to lie on the Stiefel manifold. This introduces "gauge chaos" -- the trajectory
   explores orthogonal directions even when the loss landscape is locally flat.

The prediction: near a pre-trained minimum, effect (1) causes overshooting and
effect (2) causes wandering, both of which hurt fine-tuning performance.

In [ ]:
def newton_schulz_ortho(M, n_iters=NS_ITERS):
    """Newton-Schulz iteration to approximate the orthogonal polar factor."""
    a, b, c = 3.4445, -4.7750, 2.0315
    X = M / (np.linalg.norm(M, ord='fro') + 1e-7)
    if X.shape[0] > X.shape[1]:
        X = X.T
        transposed = True
    else:
        transposed = False
    Id = np.eye(X.shape[0])
    for _ in range(n_iters):
        A = X @ X.T
        X = (a * Id + b * A + c * A @ A) @ X
    if transposed:
        X = X.T
    return X

In [ ]:
class SGDOptimizer:
    def __init__(self, weights, lr, momentum=0.9):
        self.lr = lr
        self.momentum = momentum
        self.velocity = [np.zeros_like(W) for W in weights]

    def step(self, weights, grads):
        for i in range(len(weights)):
            self.velocity[i] = self.momentum * self.velocity[i] + grads[i]
            weights[i] -= self.lr * self.velocity[i]
        return weights

In [ ]:
class MuonOptimizer:
    def __init__(self, weights, lr, momentum=0.9, ns_iters=NS_ITERS):
        self.lr = lr
        self.momentum = momentum
        self.ns_iters = ns_iters
        self.velocity = [np.zeros_like(W) for W in weights]

    def step(self, weights, grads):
        for i in range(len(weights)):
            self.velocity[i] = self.momentum * self.velocity[i] + grads[i]
            ortho_update = newton_schulz_ortho(self.velocity[i], self.ns_iters)
            weights[i] -= self.lr * ortho_update
        return weights

## Training Loop

The training loop records two quantities at each step:

1. **Loss**: 0.5 * ||f(X) - Y_target||^2 / N, measuring how well the network
   approximates the target function. This is the primary performance metric.

2. **Checkpoint distance**: ||W_t - W_checkpoint||_F, measuring how far the
   weights have moved from the pre-trained state. This is the key diagnostic
   for hypothesis H1 -- if Muon's gauge chaos causes it to wander further
   from the checkpoint, this distance will be systematically larger for Muon.

In [ ]:
def train(weights, optimizer, W_target, X, n_steps, checkpoint_weights=None):
    """Train and record loss + distance from checkpoint at each step."""
    losses = []
    distances = []
    Y_target = W_target @ X

    for step in range(n_steps):
        loss = compute_loss(weights, X, Y_target)
        losses.append(loss)

        if checkpoint_weights is not None:
            dist = checkpoint_distance(weights, checkpoint_weights)
            distances.append(dist)

        grads = compute_gradients(weights, X, Y_target)
        weights = optimizer.step(weights, grads)

    # Final loss
    loss = compute_loss(weights, X, Y_target)
    losses.append(loss)
    if checkpoint_weights is not None:
        distances.append(checkpoint_distance(weights, checkpoint_weights))

    return weights, np.array(losses), np.array(distances) if distances else None

## Main Experiment: Three-Phase Protocol

### Phase 1: Establish the Pre-Trained Basin

We train with SGD on the original target to create a high-quality checkpoint.
This checkpoint defines the "basin" that fine-tuning should exploit.
The pre-training loss must be low enough that the checkpoint represents
a genuine minimum, not a transient state.

### Phase 2: Fine-Tuning Contest (SGD vs Muon)

From the same checkpoint, we fine-tune with both optimizers on a modified
target (20% of entries changed). The key measurements:
- Which optimizer reaches lower loss? (H2)
- Which optimizer stays closer to the checkpoint? (H1)

If the gauge-fixing theory is correct, SGD should win both metrics: it
follows the local curvature to adapt to the shifted target while staying
in the pre-trained basin, whereas Muon's orthogonal updates overshoot
and wander.

### Phase 3: From-Scratch Control (SGD vs Muon)

As a control, we train from random initialization on the modified target.
This tests whether Muon's exploration advantage (which should dominate
when there is no basin to exploit) is intact. If Muon wins from scratch
but loses for fine-tuning, the exploration/exploitation asymmetry is confirmed.

In [ ]:
def run_single_experiment(run_seed):
    """Run one full experiment with a given seed."""
    rng = np.random.RandomState(run_seed)

    # ---- Targets ----
    W_target_mod = make_modified_target(W_target_original, MODIFY_FRAC, rng)

    # ---- Phase 1: Pre-train with SGD on original target ----
    weights_init = init_weights(NUM_LAYERS, rng)
    pretrain_opt = SGDOptimizer(copy_weights(weights_init), lr=PRETRAIN_LR, momentum=MOMENTUM)
    weights_pretrained, pretrain_losses, _ = train(
        copy_weights(weights_init), pretrain_opt, W_target_original, X_data, PRETRAIN_STEPS
    )
    checkpoint = copy_weights(weights_pretrained)
    print(f"    Phase 1 complete: pre-train final loss = {pretrain_losses[-1]:.6f}")
    print(f"    Pre-train loss reduction: {pretrain_losses[0]:.4f} -> {pretrain_losses[-1]:.6f} ({pretrain_losses[0]/max(pretrain_losses[-1],1e-12):.1f}x)")

    # ---- Phase 2a: Fine-tune from checkpoint with SGD ----
    ft_sgd_opt = SGDOptimizer(copy_weights(checkpoint), lr=SGD_FT_LR, momentum=MOMENTUM)
    ft_sgd_weights, ft_sgd_losses, ft_sgd_dists = train(
        copy_weights(checkpoint), ft_sgd_opt, W_target_mod, X_data, FINETUNE_STEPS,
        checkpoint_weights=checkpoint
    )
    print(f"    Phase 2a (SGD fine-tune):  final loss = {ft_sgd_losses[-1]:.6f}, dist from ckpt = {ft_sgd_dists[-1]:.4f}")

    # ---- Phase 2b: Fine-tune from checkpoint with Muon ----
    ft_muon_opt = MuonOptimizer(copy_weights(checkpoint), lr=MUON_FT_LR, momentum=MOMENTUM)
    ft_muon_weights, ft_muon_losses, ft_muon_dists = train(
        copy_weights(checkpoint), ft_muon_opt, W_target_mod, X_data, FINETUNE_STEPS,
        checkpoint_weights=checkpoint
    )
    print(f"    Phase 2b (Muon fine-tune): final loss = {ft_muon_losses[-1]:.6f}, dist from ckpt = {ft_muon_dists[-1]:.4f}")

    # ---- Phase 3a: From scratch with SGD ----
    scratch_init = init_weights(NUM_LAYERS, rng)
    scratch_sgd_opt = SGDOptimizer(copy_weights(scratch_init), lr=SGD_SCRATCH_LR, momentum=MOMENTUM)
    _, scratch_sgd_losses, _ = train(
        copy_weights(scratch_init), scratch_sgd_opt, W_target_mod, X_data, SCRATCH_STEPS
    )

    # ---- Phase 3b: From scratch with Muon ----
    scratch_muon_opt = MuonOptimizer(copy_weights(scratch_init), lr=MUON_SCRATCH_LR, momentum=MOMENTUM)
    _, scratch_muon_losses, _ = train(
        copy_weights(scratch_init), scratch_muon_opt, W_target_mod, X_data, SCRATCH_STEPS
    )
    print(f"    Phase 3 (from-scratch):   SGD final = {scratch_sgd_losses[-1]:.6f}, Muon final = {scratch_muon_losses[-1]:.6f}")

    return {
        'pretrain_final_loss': pretrain_losses[-1],
        'ft_sgd_losses': ft_sgd_losses,
        'ft_muon_losses': ft_muon_losses,
        'ft_sgd_dists': ft_sgd_dists,
        'ft_muon_dists': ft_muon_dists,
        'scratch_sgd_losses': scratch_sgd_losses,
        'scratch_muon_losses': scratch_muon_losses,
    }

## Execution, Aggregation, and Analysis

The `main()` function runs NUM_RUNS independent experiments with different
random seeds, then aggregates the results. For each run, we collect:

- Pre-training final loss (sanity check: should be low and consistent)
- Fine-tuning loss curves for SGD and Muon
- Checkpoint distance curves for SGD and Muon
- From-scratch loss curves for SGD and Muon

The aggregation computes means and standard deviations across runs,
then evaluates the three hypothesis tests (H1, H2, H3) using the
aggregated statistics.

### What to Look For in the Output

- **Loss curve snapshots**: These show the trajectory of optimization at
  key milestones. For fine-tuning, watch whether Muon's loss plateaus or
  oscillates while SGD steadily decreases. For from-scratch, watch whether
  Muon converges faster in the early phase.

- **Distance curves**: A monotonically increasing distance for Muon would
  indicate persistent wandering (gauge chaos). A distance that increases
  then plateaus would indicate that Muon finds a different basin entirely.

- **Convergence speed analysis**: Steps to reach 50% of initial loss.
  This separates "fast early progress" from "final accuracy" -- Muon might
  converge faster initially but plateau at a worse final loss.

- **Early vs late dynamics**: The ratio of loss drop in the first 50 steps
  to the last 50 steps reveals whether the optimizer is still making progress
  at the end of fine-tuning, or has stalled.

In [ ]:
def main():
    print("=" * 80)
    print("Exp 2.5: Muon worse for fine-tuning?")
    print("       Chaos prevents settling near pre-trained minimum")
    print("=" * 80)
    print()

    # Collect results across runs
    all_results = []
    for run_idx in range(NUM_RUNS):
        seed = SEED + run_idx * 137
        print(f"  Run {run_idx + 1}/{NUM_RUNS} (seed={seed})...")
        result = run_single_experiment(seed)
        all_results.append(result)

    # Aggregate
    ft_sgd_final_losses   = [r['ft_sgd_losses'][-1] for r in all_results]
    ft_muon_final_losses  = [r['ft_muon_losses'][-1] for r in all_results]
    ft_sgd_final_dists    = [r['ft_sgd_dists'][-1] for r in all_results]
    ft_muon_final_dists   = [r['ft_muon_dists'][-1] for r in all_results]
    sc_sgd_final_losses   = [r['scratch_sgd_losses'][-1] for r in all_results]
    sc_muon_final_losses  = [r['scratch_muon_losses'][-1] for r in all_results]
    pretrain_final_losses = [r['pretrain_final_loss'] for r in all_results]

    # Means and stds
    def ms(arr):
        return np.mean(arr), np.std(arr)

    pt_m, pt_s     = ms(pretrain_final_losses)
    ft_sgd_m, ft_sgd_s   = ms(ft_sgd_final_losses)
    ft_muon_m, ft_muon_s = ms(ft_muon_final_losses)
    ft_sgd_d_m, ft_sgd_d_s   = ms(ft_sgd_final_dists)
    ft_muon_d_m, ft_muon_d_s = ms(ft_muon_final_dists)
    sc_sgd_m, sc_sgd_s   = ms(sc_sgd_final_losses)
    sc_muon_m, sc_muon_s = ms(sc_muon_final_losses)

    # ---- Loss curves (averaged) ----
    print()
    print("-" * 70)
    print("LOSS CURVE SNAPSHOTS (averaged over runs)")
    print("-" * 70)

    # Fine-tuning curves
    ft_steps = len(all_results[0]['ft_sgd_losses'])
    ft_sgd_curve = np.mean([r['ft_sgd_losses'] for r in all_results], axis=0)
    ft_muon_curve = np.mean([r['ft_muon_losses'] for r in all_results], axis=0)

    print("\nFine-tuning from checkpoint (200 steps):")
    print(f"  {'Step':>6}  {'SGD loss':>12}  {'Muon loss':>12}  {'SGD dist':>12}  {'Muon dist':>12}")
    ft_sgd_dist_curve = np.mean([r['ft_sgd_dists'] for r in all_results], axis=0)
    ft_muon_dist_curve = np.mean([r['ft_muon_dists'] for r in all_results], axis=0)
    for step_idx in [0, 10, 25, 50, 100, 150, 200]:
        if step_idx < ft_steps:
            print(f"  {step_idx:>6}  {ft_sgd_curve[step_idx]:>12.6f}  "
                  f"{ft_muon_curve[step_idx]:>12.6f}  "
                  f"{ft_sgd_dist_curve[step_idx]:>12.4f}  "
                  f"{ft_muon_dist_curve[step_idx]:>12.4f}")

    # From-scratch curves
    sc_steps = len(all_results[0]['scratch_sgd_losses'])
    sc_sgd_curve = np.mean([r['scratch_sgd_losses'] for r in all_results], axis=0)
    sc_muon_curve = np.mean([r['scratch_muon_losses'] for r in all_results], axis=0)

    print("\nFrom scratch (700 steps):")
    print(f"  {'Step':>6}  {'SGD loss':>12}  {'Muon loss':>12}")
    for step_idx in [0, 50, 100, 200, 350, 500, 700]:
        if step_idx < sc_steps:
            print(f"  {step_idx:>6}  {sc_sgd_curve[step_idx]:>12.6f}  "
                  f"{sc_muon_curve[step_idx]:>12.6f}")

    # ---- Main results table ----
    print()
    print("=" * 80)
    print("MAIN RESULTS TABLE")
    print("=" * 80)
    print()
    print(f"Pre-training final loss (SGD, 500 steps): {pt_m:.6f} +/- {pt_s:.6f}")
    print()

    header = (f"  {'Scenario':<30}  {'SGD final loss':>16}  {'Muon final loss':>17}  "
              f"{'SGD dist':>14}  {'Muon dist':>14}")
    print(header)
    print("  " + "-" * (len(header) - 2))

    print(f"  {'Fine-tune (from ckpt)':<30}  "
          f"{ft_sgd_m:>10.6f}+/-{ft_sgd_s:<5.4f}  "
          f"{ft_muon_m:>10.6f}+/-{ft_muon_s:<6.4f}  "
          f"{ft_sgd_d_m:>8.4f}+/-{ft_sgd_d_s:<4.3f}  "
          f"{ft_muon_d_m:>8.4f}+/-{ft_muon_d_s:<4.3f}")

    print(f"  {'From scratch (700 steps)':<30}  "
          f"{sc_sgd_m:>10.6f}+/-{sc_sgd_s:<5.4f}  "
          f"{sc_muon_m:>10.6f}+/-{sc_muon_s:<6.4f}  "
          f"{'n/a':>14}  {'n/a':>14}")

    # ---- Verdict ----
    print()
    print("=" * 80)
    print("HYPOTHESIS TESTS")
    print("=" * 80)
    print()

    # Test 1: Fine-tuning distance
    dist_test = ft_muon_d_m > ft_sgd_d_m
    print(f"[H1] Muon wanders further from checkpoint during fine-tuning?")
    print(f"     SGD dist = {ft_sgd_d_m:.4f},  Muon dist = {ft_muon_d_m:.4f}")
    print(f"     Ratio Muon/SGD = {ft_muon_d_m / (ft_sgd_d_m + 1e-12):.2f}x")
    print(f"     --> {'CONFIRMED' if dist_test else 'REJECTED'}")
    print()

    # Test 2: Fine-tuning loss -- SGD < Muon
    ft_loss_test = ft_sgd_m < ft_muon_m
    print(f"[H2] Fine-tuning: SGD reaches lower loss than Muon?")
    print(f"     SGD loss = {ft_sgd_m:.6f},  Muon loss = {ft_muon_m:.6f}")
    if ft_sgd_m > 0:
        print(f"     Muon/SGD ratio = {ft_muon_m / ft_sgd_m:.2f}")
    print(f"     --> {'CONFIRMED' if ft_loss_test else 'REJECTED'}")
    print()

    # Test 3: From-scratch loss -- Muon < SGD
    scratch_loss_test = sc_muon_m < sc_sgd_m
    print(f"[H3] From scratch: Muon reaches lower loss than SGD?")
    print(f"     SGD loss = {sc_sgd_m:.6f},  Muon loss = {sc_muon_m:.6f}")
    if sc_sgd_m > 0:
        print(f"     SGD/Muon ratio = {sc_sgd_m / (sc_muon_m + 1e-12):.2f}")
    print(f"     --> {'CONFIRMED' if scratch_loss_test else 'REJECTED'}")
    print()

    # Overall
    all_confirmed = dist_test and ft_loss_test and scratch_loss_test
    partial = sum([dist_test, ft_loss_test, scratch_loss_test])
    print("=" * 80)
    if all_confirmed:
        print("OVERALL VERDICT: HYPOTHESIS FULLY CONFIRMED (3/3)")
        print("  Muon is worse for fine-tuning (chaos prevents settling)")
        print("  but better from scratch (exploration advantage).")
    elif partial >= 2:
        print(f"OVERALL VERDICT: HYPOTHESIS PARTIALLY CONFIRMED ({partial}/3)")
        if not dist_test:
            print("  Surprise: Muon did NOT wander further from checkpoint.")
        if not ft_loss_test:
            print("  Surprise: Muon actually fine-tuned to LOWER loss than SGD.")
        if not scratch_loss_test:
            print("  Surprise: SGD actually trained from scratch to LOWER loss.")
    else:
        print(f"OVERALL VERDICT: HYPOTHESIS REJECTED ({partial}/3)")
        if not ft_loss_test:
            print("  Key finding: Muon fine-tunes as well or better than SGD.")
        if not scratch_loss_test:
            print("  Key finding: SGD trains from scratch as well or better.")
    print("=" * 80)

    # ---- Additional analysis: convergence speed ----
    print()
    print("-" * 70)
    print("ADDITIONAL ANALYSIS: Convergence Speed")
    print("-" * 70)

    # For fine-tuning: steps to reach 50% of initial loss
    def steps_to_threshold(losses, frac=0.5):
        threshold = losses[0] * frac
        for i, l in enumerate(losses):
            if l <= threshold:
                return i
        return len(losses)

    ft_sgd_half = np.mean([steps_to_threshold(r['ft_sgd_losses']) for r in all_results])
    ft_muon_half = np.mean([steps_to_threshold(r['ft_muon_losses']) for r in all_results])
    sc_sgd_half = np.mean([steps_to_threshold(r['scratch_sgd_losses']) for r in all_results])
    sc_muon_half = np.mean([steps_to_threshold(r['scratch_muon_losses']) for r in all_results])

    print(f"\nSteps to reach 50% of initial loss:")
    print(f"  Fine-tune SGD:      {ft_sgd_half:.1f}")
    print(f"  Fine-tune Muon:     {ft_muon_half:.1f}")
    print(f"  From-scratch SGD:   {sc_sgd_half:.1f}")
    print(f"  From-scratch Muon:  {sc_muon_half:.1f}")

    # ---- Early vs Late fine-tuning comparison ----
    print()
    print("-" * 70)
    print("FINE-TUNING DYNAMICS: Early vs Late")
    print("-" * 70)
    # Compare loss reduction in first 50 steps vs last 50 steps
    for name, curve in [("SGD", ft_sgd_curve), ("Muon", ft_muon_curve)]:
        early_drop = curve[0] - curve[50]
        late_drop = curve[150] - curve[200] if len(curve) > 200 else curve[-51] - curve[-1]
        print(f"  {name}: early drop (0-50) = {early_drop:.6f},  "
              f"late drop (150-200) = {late_drop:.6f},  "
              f"ratio = {early_drop / (abs(late_drop) + 1e-12):.1f}x")

    print()
    print("Experiment complete.")

## Run the Experiment

Executing all phases across all random seeds. Expected runtime: a few seconds
(deep linear networks with DIM=32 are computationally lightweight).

In [ ]:
if __name__ == "__main__":
    main()

## Interpretation Guide

### Reading the Results

**If all three hypotheses are CONFIRMED (3/3):**

This is the strongest evidence for the gauge-fixing interpretation. It means:
- Muon's Newton-Schulz orthogonalization genuinely injects trajectory chaos
  (evidenced by larger checkpoint distances).
- This chaos is harmful for fine-tuning (higher final loss) but helpful
  for training from scratch (lower final loss).
- The exploration/exploitation asymmetry is exactly what the RG gauge-fixing
  theory predicts: gauge-fixing helps when symmetry must be broken (exploration)
  but hurts when a particular symmetry-broken state must be preserved (exploitation).

**If H1 is confirmed but H2 is rejected (Muon wanders but still fine-tunes well):**

This would suggest that Muon finds a DIFFERENT but equally good minimum -- the
wandering is not pathological but exploratory even in the fine-tuning regime.
This would weaken the "chaos hurts fine-tuning" narrative and suggest that
Muon's gauge-fixing finds alternative basins that are just as good.

**If H3 is rejected (SGD beats Muon from scratch):**

This would undermine the entire exploration advantage story. Possible causes:
- The learning rate ratio is unfavorable for Muon
- The problem is too easy (DIM=32) for exploration to matter
- The deep linear setting does not adequately capture the nonlinear dynamics
  where Muon's advantage is typically observed

### Connections to Broader Theory

This experiment connects to several broader themes:

1. **Exploration vs exploitation in optimization**: The tension between
   exploring the loss landscape (finding good basins) and exploiting a known
   basin (fine-tuning within it) is fundamental to optimization theory.

2. **Gauge symmetry and optimization dynamics**: If Muon's gauge-fixing
   mechanism is regime-dependent, this implies that the optimal gauge choice
   depends on the optimization phase -- a prediction that could inform
   adaptive optimizer design (e.g., use Muon for pre-training, switch to
   SGD for fine-tuning).

3. **Lyapunov exponents and optimizer stability**: The checkpoint distance
   growth rate is related to the Lyapunov exponent of the optimizer dynamics.
   A higher Lyapunov exponent near a fixed point means the optimizer is
   more "chaotic" -- it amplifies small perturbations rather than damping them.

4. **Practical implications for transfer learning**: If confirmed, this
   result suggests that practitioners should NOT use Muon for fine-tuning
   pre-trained models, even if Muon was used for pre-training. The optimal
   strategy would be a two-phase approach: Muon for exploration, then SGD
   for exploitation.

## Conclusions and Next Steps

### Summary of Findings

This experiment tested the prediction that Muon's gauge-direction chaos
creates an asymmetry between exploration (training from scratch) and
exploitation (fine-tuning). The three hypothesis tests (H1: distance,
H2: fine-tuning loss, H3: from-scratch loss) provide a comprehensive
assessment of this prediction.

### Limitations

- **Deep linear networks**: The gauge symmetry GL(n) is exact in deep
  linear networks but only approximate in nonlinear networks with ReLU/GELU.
  Results may not directly transfer to practical architectures.
- **Small scale**: DIM=32 may not capture phenomena that emerge at larger scale.
- **Fixed modification fraction**: 20% target modification is a single point
  on a continuum. The transition from "fine-tuning is better" to "from-scratch
  is better" likely depends on the modification fraction.
- **Learning rate sensitivity**: The relative performance of SGD vs Muon
  depends on the learning rate ratio, which is not tuned here.

### Suggested Follow-Up Experiments

1. **Sweep modification fraction** (5%, 10%, 20%, 50%, 80%) to map the
   crossover point where fine-tuning stops being beneficial.
2. **Sweep learning rates** to verify that the results are robust to
   hyperparameter choices.
3. **Measure Lyapunov exponents directly** by tracking the divergence of
   nearby trajectories during fine-tuning.
4. **Nonlinear networks**: Repeat with ReLU activations to test whether
   the gauge-fixing mechanism transfers to the approximate-symmetry regime.
5. **Hybrid optimizer**: Use Muon for the first N steps (exploration) then
   switch to SGD (exploitation) to test whether a two-phase strategy
   outperforms either optimizer alone.